
# Swell Forecast Models Report (per-model extraction + pixel visualization)

This notebook is generated from `swell_forecast_extractor.py` and is designed to:

- Run **each model individually** (Open‑Meteo Marine API models and CDIP models).
- Document model characteristics, origins, and variables extracted.
- Show **where values are taken from** by mapping sites and the chosen grid cell / buoy.
- For each model, render a **Folium map** with a basemap and an example raster/field for one instant:
  - **CDIP MOP CA Grid**: true model grid (OPeNDAP) raster overlay.
  - **Open‑Meteo models**: an **approximate field** built by sampling a coarse grid of points within the bbox.
  - **CDIP station forecasts**: point-based (buoy location) map.

Finally, it produces a faceted (by site) time-series plot of **swell height vs time** across all models.

> **BBox used (lon_min, lon_max, lat_max, lat_min):**
> `(-120.77405702695488, -118.99968426997366, 34.73458619896702, 33.80267649583392)`


In [ ]:

# If needed, install dependencies (uncomment and run once)
# %pip install -q pandas numpy requests xarray netCDF4 matplotlib folium pillow tqdm


In [ ]:

import os
import sys
import math
import json
import time
from datetime import datetime, timedelta

import numpy as np
import pandas as pd

# Plotting
import matplotlib.pyplot as plt

# Mapping
import folium
from folium.raster_layers import ImageOverlay

# Image helpers
from PIL import Image
import io

from tqdm.auto import tqdm


In [ ]:

# Import the extractor script
# (expects swell_forecast_extractor.py in the same folder as this notebook)

HERE = os.getcwd()
if HERE not in sys.path:
    sys.path.insert(0, HERE)

import swell_forecast_extractor as sfe

print('Loaded swell_forecast_extractor from:', sfe.__file__)
print('Open‑Meteo model keys:', list(sfe.MARINE_MODEL_CONFIGS.keys()))


In [ ]:

# -----------------------------
# Inputs
# -----------------------------

BBOX = (-120.77405702695488, -118.99968426997366, 34.73458619896702, 33.80267649583392)
lon_min, lon_max, lat_max, lat_min = BBOX

# Optional: a CSV with columns: site_name, latitude, longitude
SITES_CSV = os.environ.get('SITES_CSV', 'sites.csv')

if os.path.exists(SITES_CSV):
    sites_df = pd.read_csv(SITES_CSV)
    if 'site_name' not in sites_df.columns:
        sites_df['site_name'] = [f"Site_{i+1}" for i in range(len(sites_df))]
else:
    # Fallback example sites inside the bbox
    sites_df = pd.DataFrame([
        {'site_name': 'Example_Site_A', 'latitude': 34.000, 'longitude': -120.00},
        {'site_name': 'Example_Site_B', 'latitude': 33.950, 'longitude': -119.50},
    ])

sites_df



## Model catalog

Open‑Meteo models come from `MARINE_MODEL_CONFIGS` in the script.

CDIP models are discovered at runtime:
- `cdip_mop_ca_grid_forecast`
- `cdip_mop_forecast_<station_id>` for each closest active buoy station.


In [ ]:

open_meteo_keys = list(sfe.MARINE_MODEL_CONFIGS.keys())
open_meteo_table = (
    pd.DataFrame.from_dict(sfe.MARINE_MODEL_CONFIGS, orient='index')
      .reset_index()
      .rename(columns={'index': 'model_key'})
      [['model_key','display_name','resolution','forecast_days','description','has_secondary_swell','has_tertiary_swell']]
)
open_meteo_table


In [ ]:

# Discover closest active CDIP stations for these sites (so we can treat each station forecast as its own model)

def discover_cdip_station_models(sites_df: pd.DataFrame, n_closest: int = 2, active_within_days: int = 7):
    station_models = {}
    per_site = {}

    active = sfe.fetch_active_cdip_stations(active_within_days=active_within_days)
    if active.empty:
        print('No active CDIP stations returned (ERDDAP). Station models will be skipped.')
        return station_models, per_site, active

    for _, row in sites_df.iterrows():
        site = row['site_name']
        lat = float(row['latitude']); lon = float(row['longitude'])
        closest = sfe.find_closest_cdip_stations(lat, lon, n=n_closest, active_within_days=active_within_days)
        per_site[site] = closest
        for st in closest:
            sid = str(st['station_id']).zfill(3)
            station_models[sid] = st

    return station_models, per_site, active

cdip_station_models, cdip_closest_per_site, cdip_active_df = discover_cdip_station_models(sites_df, n_closest=2)

print('Discovered station forecast models:', [f"cdip_mop_forecast_{sid}" for sid in sorted(cdip_station_models.keys())])
cdip_active_df.head()


In [ ]:

# Full list of per-model keys we will document
all_model_keys = (
    open_meteo_keys
    + ['cdip_mop_ca_grid_forecast']
    + [f"cdip_mop_forecast_{sid}" for sid in sorted(cdip_station_models.keys())]
)

all_model_keys


In [ ]:

# -----------------------------
# Utilities
# -----------------------------

CACHE_DIR = os.environ.get('SWELL_NOTEBOOK_CACHE', 'cache')
os.makedirs(CACHE_DIR, exist_ok=True)


def parse_iso_dt(x):
    try:
        return pd.to_datetime(x, errors='coerce')
    except Exception:
        return pd.NaT


def render_array_to_png(arr2d, vmin=None, vmax=None):
    # Returns PNG bytes for folium ImageOverlay
    import matplotlib.pyplot as plt

    fig = plt.figure(figsize=(6, 6), dpi=160)
    ax = fig.add_axes([0, 0, 1, 1])
    ax.axis('off')

    im = ax.imshow(arr2d, origin='lower', vmin=vmin, vmax=vmax)

    buf = io.BytesIO()
    fig.savefig(buf, format='png', transparent=True)
    plt.close(fig)
    buf.seek(0)
    return buf.getvalue()


def add_basemaps(m):
    folium.TileLayer(
        tiles='OpenStreetMap',
        name='OpenStreetMap',
        overlay=False,
        control=True,
    ).add_to(m)

    folium.TileLayer(
        tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
        attr='Esri',
        name='Esri.WorldImagery',
        overlay=False,
        control=True,
    ).add_to(m)


def bbox_center(bbox):
    lon_min, lon_max, lat_max, lat_min = bbox
    return (lat_max + lat_min) / 2.0, (lon_min + lon_max) / 2.0


def make_base_map(bbox, zoom_start=9):
    center_lat, center_lon = bbox_center(bbox)
    m = folium.Map(location=[center_lat, center_lon], zoom_start=zoom_start, tiles=None)
    add_basemaps(m)
    folium.LayerControl(collapsed=False).add_to(m)

    # bbox outline
    folium.Rectangle(
        bounds=[[bbox[3], bbox[0]], [bbox[2], bbox[1]]],
        color='white',
        weight=2,
        fill=False,
        tooltip='BBox'
    ).add_to(m)

    return m


def add_sites(m, sites_df):
    for _, r in sites_df.iterrows():
        folium.CircleMarker(
            location=[float(r['latitude']), float(r['longitude'])],
            radius=5,
            weight=2,
            fill=True,
            fill_opacity=0.8,
            tooltip=str(r['site_name'])
        ).add_to(m)


def approx_pixel_bounds(lat, lon, resolution_km):
    # Rough conversion km -> degrees (good enough for a visualization box)
    dlat = (resolution_km / 111.0) / 2.0
    dlon = (resolution_km / (111.0 * math.cos(math.radians(lat)) + 1e-9)) / 2.0
    return [[lat - dlat, lon - dlon], [lat + dlat, lon + dlon]]


def open_meteo_point_cell_center(lat, lon, model_key):
    # One quick request so we can read the API-selected grid cell center
    local_date, local_now, tz_name = sfe.get_local_date(lat, lon)
    start_date = local_date.strftime('%Y-%m-%d')
    end_date = (local_date + timedelta(days=1)).strftime('%Y-%m-%d')

    resp = sfe.fetch_marine_forecast(lat, lon, model_key, start_date, end_date, tz_name)
    if isinstance(resp, dict) and 'latitude' in resp and 'longitude' in resp:
        return float(resp['latitude']), float(resp['longitude']), tz_name
    return float(lat), float(lon), tz_name


def sample_open_meteo_field(bbox, model_key, date_str, grid_n=7, sleep_s=0.0):
    # Build a coarse field by sampling points within the bbox.
    # Cached on disk because it can take a minute.
    cache_path = os.path.join(CACHE_DIR, f"open_meteo_field_{model_key}_{date_str}_n{grid_n}.npz")
    if os.path.exists(cache_path):
        z = np.load(cache_path)
        return z['field'], z['lats'], z['lons']

    lon_min, lon_max, lat_max, lat_min = bbox
    lats = np.linspace(lat_min, lat_max, grid_n)
    lons = np.linspace(lon_min, lon_max, grid_n)

    field = np.full((grid_n, grid_n), np.nan, dtype=float)

    # For performance, request just one day and read the first hour value
    for i, la in enumerate(tqdm(lats, desc=f"Sampling {model_key} lat")):
        for j, lo in enumerate(lons):
            # Use timezone=auto; single day
            url = sfe.build_marine_api_url(
                float(la), float(lo),
                sfe.MARINE_MODEL_CONFIGS[model_key]['model_param'],
                date_str, date_str,
                'auto',
                include_secondary=False,
                include_tertiary=False,
            )
            try:
                r = sfe.requests.get(url, timeout=30)
                r.raise_for_status()
                js = r.json()
                wh = None
                if 'hourly' in js and 'wave_height' in js['hourly']:
                    wh = js['hourly']['wave_height'][0]
                if wh is None and 'hourly' in js and 'swell_wave_height' in js['hourly']:
                    wh = js['hourly']['swell_wave_height'][0]
                field[i, j] = np.nan if wh is None else float(wh)
            except Exception:
                field[i, j] = np.nan

            if sleep_s:
                time.sleep(sleep_s)

    np.savez(cache_path, field=field, lats=lats, lons=lons)
    return field, lats, lons


def cdip_grid_subset_to_field(bbox, wave_time_index=0):
    # True CDIP grid raster for waveHs at an instant
    import xarray as xr

    cdip_url = 'https://thredds.cdip.ucsd.edu/thredds/dodsC/cdip/model/MOP_grids/CA_0.01_forecast.nc'
    ds = xr.open_dataset(cdip_url)

    lats = ds['metaLatitude'].values
    lons = ds['metaLongitude'].values

    lon_min, lon_max, lat_max, lat_min = bbox

    lat_mask = (lats >= lat_min) & (lats <= lat_max)
    lon_mask = (lons >= lon_min) & (lons <= lon_max)

    lat_idx = np.where(lat_mask)[0]
    lon_idx = np.where(lon_mask)[0]

    sub = ds['waveHs'].isel(waveTime=wave_time_index, metaLatitude=lat_idx, metaLongitude=lon_idx).values

    out = {
        'field': sub,
        'lats': lats[lat_idx],
        'lons': lons[lon_idx],
        'times': ds['waveTime'].values,
    }

    ds.close()
    return out


def cdip_nearest_grid_cell(lat, lon):
    import xarray as xr

    cdip_url = 'https://thredds.cdip.ucsd.edu/thredds/dodsC/cdip/model/MOP_grids/CA_0.01_forecast.nc'
    ds = xr.open_dataset(cdip_url)

    lats = ds['metaLatitude'].values
    lons = ds['metaLongitude'].values

    lat_idx = int(np.argmin(np.abs(lats - float(lat))))
    lon_idx = int(np.argmin(np.abs(lons - float(lon))))

    nearest_lat = float(lats[lat_idx])
    nearest_lon = float(lons[lon_idx])

    # Approx bounds from adjacent coords when possible
    def mid(a, b):
        return (a + b) / 2.0

    if lat_idx == 0:
        lat0 = nearest_lat - (lats[1] - lats[0]) / 2.0
    else:
        lat0 = mid(lats[lat_idx-1], lats[lat_idx])

    if lat_idx == len(lats)-1:
        lat1 = nearest_lat + (lats[-1] - lats[-2]) / 2.0
    else:
        lat1 = mid(lats[lat_idx], lats[lat_idx+1])

    if lon_idx == 0:
        lon0 = nearest_lon - (lons[1] - lons[0]) / 2.0
    else:
        lon0 = mid(lons[lon_idx-1], lons[lon_idx])

    if lon_idx == len(lons)-1:
        lon1 = nearest_lon + (lons[-1] - lons[-2]) / 2.0
    else:
        lon1 = mid(lons[lon_idx], lons[lon_idx+1])

    ds.close()

    bounds = [[float(lat0), float(lon0)], [float(lat1), float(lon1)]]
    return nearest_lat, nearest_lon, bounds


## Per-model extraction

In [ ]:


def run_open_meteo_model(sites_df: pd.DataFrame, model_key: str) -> pd.DataFrame:
    df = sfe.extract_swell_forecasts(
        sites_df.copy(),
        models_to_use=[model_key],
        include_cdip=False,
    )
    return df


def run_cdip_grid_model(sites_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, r in sites_df.iterrows():
        site = r['site_name']
        lat = float(r['latitude']); lon = float(r['longitude'])
        local_date, local_now, tz_name = sfe.get_local_date(lat, lon)
        out = sfe.fetch_cdip_mop_grid_forecast(lat, lon, site_name=site, local_date=local_date)
        rows.extend(out)
    return pd.DataFrame(rows)


def run_cdip_station_model(sites_df: pd.DataFrame, station_id: str, per_site_closest: dict) -> pd.DataFrame:
    sid = str(station_id).zfill(3)
    rows = []

    # Lookup station name/coords from the active list
    active = sfe.fetch_active_cdip_stations(active_within_days=7)
    st_row = active[active['station_id'].astype(str).str.zfill(3) == sid]
    st_name = None
    if len(st_row) > 0:
        st_name = str(st_row.iloc[0].get('metaStationName', sid))

    for _, r in sites_df.iterrows():
        site = r['site_name']
        lat = float(r['latitude']); lon = float(r['longitude'])

        # Only fetch for sites where this station is among the n-closest (matches extractor behavior)
        closest_list = per_site_closest.get(site, [])
        match = [c for c in closest_list if str(c['station_id']).zfill(3) == sid]
        if not match:
            continue
        dist_km = float(match[0]['distance_km'])
        if st_name is None:
            st_name = str(match[0].get('name', sid))

        local_date, local_now, tz_name = sfe.get_local_date(lat, lon)
        out = sfe.fetch_cdip_station_forecast(
            station_id=sid,
            station_name=st_name,
            site_name=site,
            site_lat=lat,
            site_lon=lon,
            local_date=local_date,
            distance_km=dist_km,
        )
        rows.extend(out)

    return pd.DataFrame(rows)


In [ ]:


def build_model_map(model_key: str, sites_df: pd.DataFrame, bbox=BBOX, grid_n=7):
    m = make_base_map(bbox)
    add_sites(m, sites_df)

    # Choose an example date for sampling (today in UTC)
    date_str = datetime.utcnow().strftime('%Y-%m-%d')

    if model_key == 'cdip_mop_ca_grid_forecast':
        try:
            sub = cdip_grid_subset_to_field(bbox, wave_time_index=0)
            field = sub['field']

            # Render to png and overlay
            png = render_array_to_png(field)
            img = Image.open(io.BytesIO(png))

            bounds = [[bbox[3], bbox[0]], [bbox[2], bbox[1]]]
            ImageOverlay(image=img, bounds=bounds, opacity=0.6, name='waveHs (CDIP grid)')\
                .add_to(m)

            # Pixel boxes for each site
            for _, r in sites_df.iterrows():
                nlat, nlon, pb = cdip_nearest_grid_cell(r['latitude'], r['longitude'])
                folium.CircleMarker(
                    location=[nlat, nlon], radius=4, weight=2,
                    fill=True, fill_opacity=0.8,
                    tooltip=f"CDIP grid cell center for {r['site_name']}"
                ).add_to(m)
                folium.Rectangle(bounds=pb, color='yellow', weight=2, fill=False).add_to(m)

        except Exception as e:
            folium.Marker(location=bbox_center(bbox), tooltip=f"CDIP grid overlay failed: {e}").add_to(m)

    elif model_key.startswith('cdip_mop_forecast_'):
        # Station point map (no raster)
        sid = model_key.split('_')[-1]
        st = cdip_station_models.get(sid)
        if st:
            folium.Marker(
                location=[float(st['lat']), float(st['lon'])],
                tooltip=f"CDIP buoy {sid}: {st.get('name','')}",
                icon=folium.Icon(color='green', icon='info-sign'),
            ).add_to(m)

            for _, r in sites_df.iterrows():
                # draw only if station is among closest
                site = r['site_name']
                closest = cdip_closest_per_site.get(site, [])
                if any(str(c['station_id']).zfill(3) == sid for c in closest):
                    folium.PolyLine(
                        locations=[[float(r['latitude']), float(r['longitude'])], [float(st['lat']), float(st['lon'])]],
                        weight=2,
                        opacity=0.8,
                        tooltip=f"{site} -> buoy {sid}"
                    ).add_to(m)

    else:
        # Open‑Meteo model: approximate raster by sampling a coarse grid
        try:
            field, lats, lons = sample_open_meteo_field(bbox, model_key, date_str, grid_n=grid_n, sleep_s=0.0)
            png = render_array_to_png(field)
            img = Image.open(io.BytesIO(png))

            bounds = [[bbox[3], bbox[0]], [bbox[2], bbox[1]]]
            ImageOverlay(image=img, bounds=bounds, opacity=0.55, name='Sampled wave height')\
                .add_to(m)

            # Pixel boxes for each site (API-selected grid cell center)
            cfg = sfe.MARINE_MODEL_CONFIGS.get(model_key, {})
            res = cfg.get('resolution', '~25km')
            # Parse resolution like "~9km" -> 9
            try:
                res_km = float(''.join([c for c in str(res) if (c.isdigit() or c=='.')]))
            except Exception:
                res_km = 25.0

            for _, r in sites_df.iterrows():
                clat, clon, tz = open_meteo_point_cell_center(float(r['latitude']), float(r['longitude']), model_key)
                folium.CircleMarker(
                    location=[clat, clon], radius=4, weight=2,
                    fill=True, fill_opacity=0.8,
                    tooltip=f"API-selected cell center for {r['site_name']}"
                ).add_to(m)
                pb = approx_pixel_bounds(clat, clon, res_km)
                folium.Rectangle(bounds=pb, color='yellow', weight=2, fill=False).add_to(m)

        except Exception as e:
            folium.Marker(location=bbox_center(bbox), tooltip=f"Open‑Meteo field sampling failed: {e}").add_to(m)

    return m


In [ ]:
# -----------------------------
# Run + document each model
# -----------------------------

per_model_results = {}

for model_key in all_model_keys:
    print("\n" + "="*80)
    print("MODEL:", model_key)

    cache_csv = os.path.join(CACHE_DIR, f"timeseries_{model_key}.csv")
    df_m = None
    if os.path.exists(cache_csv):
        print("  -> loading cached time series:", cache_csv)
        df_m = pd.read_csv(cache_csv)
    else:
        if model_key in open_meteo_keys:
            df_m = run_open_meteo_model(sites_df, model_key)
        elif model_key == 'cdip_mop_ca_grid_forecast':
            df_m = run_cdip_grid_model(sites_df)
        elif model_key.startswith('cdip_mop_forecast_'):
            station_id = model_key.split('_')[-1]
            df_m = run_cdip_station_model(sites_df, station_id)
        else:
            df_m = pd.DataFrame()

        if isinstance(df_m, pd.DataFrame) and not df_m.empty:
            df_m.to_csv(cache_csv, index=False)
            print("  -> cached time series:", cache_csv)

    per_model_results[model_key] = df_m

    # Summary
    if df_m is None or df_m.empty:
        print("  No rows")
    else:
        print("  Rows:", len(df_m))
        print("  Columns:", list(df_m.columns))
        if 'wave_height_m' in df_m.columns:
            nn = df_m['wave_height_m'].notna().sum()
            print("  Non-missing wave_height_m:", nn)

    # Map (example raster/point overlay)
    try:
        m = build_model_map(model_key, sites_df, bbox=BBOX, grid_n=7)
        show_folium(m)
    except Exception as e:
        print("  Map failed:", e)


## Combined plot: swell height vs time (faceted by site)

In [ ]:

# Combine all results
all_df = pd.concat([df for df in per_model_results.values() if isinstance(df, pd.DataFrame) and len(df) > 0], ignore_index=True)

if all_df.empty:
    raise ValueError('No data returned. Check connectivity and installed dependencies (xarray/netCDF4)')

# Parse time
all_df['timestamp_dt'] = pd.to_datetime(all_df['timestamp_local'], errors='coerce')

# Use swell_height_m if available, else wave_height_m
all_df['swell_height_plot_m'] = all_df['swell_height_m']
mask = all_df['swell_height_plot_m'].isna() & all_df['wave_height_m'].notna()
all_df.loc[mask, 'swell_height_plot_m'] = all_df.loc[mask, 'wave_height_m']

all_df = all_df.dropna(subset=['timestamp_dt']).sort_values(['site_name','timestamp_dt'])

all_df[['site_name','model_key','timestamp_dt','swell_height_plot_m']].head()


In [ ]:

# Faceted line plot (one subplot per site)

sites = sorted(all_df['site_name'].dropna().unique().tolist())

n = len(sites)
fig, axes = plt.subplots(nrows=n, ncols=1, figsize=(12, max(3, 3*n)), sharex=True)
if n == 1:
    axes = [axes]

for ax, site in zip(axes, sites):
    sdf = all_df[all_df['site_name'] == site]

    for model_key, g in sdf.groupby('model_key'):
        g = g.sort_values('timestamp_dt')
        ax.plot(g['timestamp_dt'], g['swell_height_plot_m'], label=model_key)

    ax.set_title(site)
    ax.set_ylabel('Swell height (m)')
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Time')
axes[0].legend(loc='upper right', ncol=1, fontsize=9)

plt.tight_layout()
plt.show()
